In [ ]:
from google.colab import files
files.upload()  # upload your kaggle.json file

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
print("Done!")

In [ ]:
os.makedirs('/content/dataset1', exist_ok=True)
os.system('kaggle datasets download -d nameera06/network-traffic --unzip -p /content/dataset1')
print("Downloaded!")

In [ ]:
import os
for f in os.listdir('/content/dataset1'):
    print(f)

In [ ]:
os.makedirs('/content/dataset2', exist_ok=True)
os.system('kaggle datasets download -d dhoogla/cicdarknet2020 --unzip -p /content/dataset2')
print("Downloaded!")

In [ ]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('/content/dataset1/Midterm_02_group.csv')
print("Shape:", df1.shape)
print("Columns:", df1.columns.tolist())
print("dtypes:\n", df1.dtypes)

In [ ]:
# Keep only numeric columns
df1_numeric = df1.select_dtypes(include=[np.number])

# Replace inf and fill missing with 0
df1_numeric = df1_numeric.replace([np.inf, -np.inf], np.nan)
df1_numeric = df1_numeric.fillna(0)

print("Shape after cleaning:", df1_numeric.shape)
print("Columns:", df1_numeric.columns.tolist())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Check if Label column exists
if 'Label' in df1.columns:
    df1_numeric['Label'] = le.fit_transform(df1['Label'])
else:
    # Use last column as label
    label_col = df1.columns[-1]
    df1_numeric['Label'] = le.fit_transform(df1[label_col])

print("Classes found:", le.classes_)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

X = df1_numeric.drop('Label', axis=1)
y = df1_numeric['Label']

print("Number of features:", X.shape[1])

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
np.save('/content/X_train.npy', X_train)
np.save('/content/X_test.npy', X_test)
np.save('/content/y_train.npy', y_train)
np.save('/content/y_test.npy', y_test)
print("All saved! Share these 4 files with Student 2")

In [ ]:
# Check all columns including non-numeric
print("ALL columns in original file:")
print(df1.dtypes)
print("\nSample data:")
print(df1.head(3))

In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# Encode Protocol (TCP, UDP, RARP etc)
le_protocol = LabelEncoder()
df1['Protocol_enc'] = le_protocol.fit_transform(df1['Protocol'].fillna('Unknown'))

# Encode Source and Destination IPs
le_src = LabelEncoder()
le_dst = LabelEncoder()
df1['Source_enc'] = le_src.fit_transform(df1['Source'].fillna('Unknown'))
df1['Destination_enc'] = le_dst.fit_transform(df1['Destination'].fillna('Unknown'))

# Extract port numbers from Info column
df1['src_port'] = df1['Info'].str.extract(r'(\d+)\s*>').astype(float).fillna(0)
df1['dst_port'] = df1['Info'].str.extract(r'>\s*(\d+)').astype(float).fillna(0)

# Build feature matrix
X = df1[['No.', 'Time', 'Length', 'Protocol_enc', 'Source_enc', 'Destination_enc', 'src_port', 'dst_port']]
X = X.fillna(0)

# Use Protocol as label (classifying traffic type)
le_label = LabelEncoder()
y = le_label.fit_transform(df1['Protocol'].fillna('Unknown'))

print("Features shape:", X.shape)
print("Classes:", le_label.classes_)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
import numpy as np
np.save('/content/X_train.npy', X_train)
np.save('/content/X_test.npy', X_test)
np.save('/content/y_train.npy', y_train)
np.save('/content/y_test.npy', y_test)
print("All saved! Share these 4 files with Student 2")

In [ ]:
from google.colab import files
files.download('/content/X_train.npy')
files.download('/content/X_test.npy')
files.download('/content/y_train.npy')
files.download('/content/y_test.npy')

In [ ]:
from google.colab import files
files.download('/content/X_train.npy')

In [ ]:
files.download('/content/X_test.npy')

In [ ]:
files.download('/content/y_train.npy')

In [ ]:
from google.colab import files
files.upload()
# Upload: X_train.npy, X_test.npy, y_train.npy, y_test.npy

In [ ]:
import numpy as np

X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')

print("Data loaded!")
print("X_train shape:", X_train.shape)
print("y_train unique classes:", len(np.unique(y_train)))

In [ ]:
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
print("Reshaped X_train:", X_train.shape)
print("Reshaped X_test:", X_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                     MultiHeadAttention, LayerNormalization,
                                     GlobalAveragePooling1D)

def build_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    lstm_out = LSTM(64, return_sequences=True)(inputs)
    attention_out = MultiHeadAttention(num_heads=4, key_dim=16)(lstm_out, lstm_out)
    attention_out = LayerNormalization()(attention_out + lstm_out)
    pooled = GlobalAveragePooling1D()(attention_out)
    dropped = Dropout(0.3)(pooled)
    outputs = Dense(num_classes, activation='softmax')(dropped)
    model = Model(inputs, outputs)
    return model

num_classes = len(__import__('numpy').unique(y_train))
model = build_model((X_train.shape[1], X_train.shape[2]), num_classes)
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print("Training done!")

In [ ]:
import pickle

model.save('/content/trained_model1.h5')

with open('/content/history1.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print("Model and history saved!")

In [ ]:
import numpy as np
import pickle
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, classification_report)

# Load history
with open('/content/history1.pkl', 'rb') as f:
    history = pickle.load(f)

# Get predictions
X_test_eval = X_test
y_pred_prob = model.predict(X_test_eval)
y_pred = np.argmax(y_pred_prob, axis=1)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
print("F1-Score :", f1_score(y_test, y_pred, average='weighted'))
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Dataset 1')
plt.savefig('/content/confusion_matrix.png')
plt.show()

# Accuracy and Loss Curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history['accuracy'], label='Train Accuracy')
plt.plot(history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('/content/accuracy_loss_curves.png')
plt.show()

In [ ]:
from google.colab import files
files.download('/content/confusion_matrix.png')
files.download('/content/accuracy_loss_curves.png')
files.download('/content/trained_model1.h5')
files.download('/content/history1.pkl')

In [ ]:
import json

with open('/content/drive/MyDrive/student1_preprocessing.ipynb', 'r') as f:
    nb = json.load(f)

# Clear all outputs
for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        cell['outputs'] = []
        cell['execution_count'] = None

with open('/content/cleaned_notebook.ipynb', 'w') as f:
    json.dump(nb, f)

print("Done!")

from google.colab import files
files.download('/content/cleaned_notebook.ipynb')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-name', '*.ipynb'],
                      capture_output=True, text=True)
print(result.stdout)